In [ ]:
#konfiguracija
import os, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

#sutvarkom seed kad rezultatai butu vienodi po kiekvieno paleidimo
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

#datos: nuo 2020-04-01 iki 2025 pabaigos mokom, 2026 q1 testas
TRAIN_START  = pd.Timestamp("2020-04-01")
EFF_START    = pd.Timestamp("2021-03-31")   #nuo siol turim visus lag pozymius
BLIND_START  = pd.Timestamp("2026-01-01")
BLIND_END    = pd.Timestamp("2026-03-31")

#failai
CSV_PATH    = "duomenys.csv"
LOG_PATH    = "XGBoost_Zurnalas.csv"
PARAMS_PATH = "XGBoost_best_params.json"

#paskutine diena su faktais (kitos savaites prognoze prasidesi nuo cia + 1d)
_raw_csv = pd.read_csv(CSV_PATH)
_raw_csv.columns = ['Data'] + list(_raw_csv.columns[1:])
_raw_csv['Data'] = pd.to_datetime(_raw_csv['Data'], dayfirst=True)
_col_b = _raw_csv.iloc[:, 1]
_mask = _col_b.notna() & (_col_b != '') & (_col_b != 0)
TRAIN_END = pd.Timestamp(_raw_csv.loc[_mask, 'Data'].max())
del _raw_csv, _col_b, _mask
print(f'Mokymo periodas: {TRAIN_START.date()} -> {TRAIN_END.date()}')

#kokius lag zingsnius naudosim
LAGS = (1, 2, 3, 7, 14, 28, 364)

#metrikos
def mae(y, yhat):  return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))
def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat))**2)))
def mape(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    mask = y != 0
    return float(np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100) if mask.any() else np.nan
def mase(y, yhat, y_hist, m=7):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    y_hist = np.asarray(y_hist, dtype=float)
    denom = float(np.mean(np.abs(y_hist[m:] - y_hist[:-m])))
    return float(np.mean(np.abs(y - yhat)) / denom) if denom > 0 else np.nan

def savaites_nr_metuose(data: pd.Timestamp) -> int:
    #pirma savaite = nuo sausio 1 iki pirmo sekmadienio (gali but trumpa)
    #toliau viskas pir-sek
    jan1 = pd.Timestamp(year=data.year, month=1, day=1)
    pirmas_sekm = jan1 + pd.Timedelta(days=(6 - jan1.dayofweek) % 7)
    if data <= pirmas_sekm:
        return 1
    return 2 + (data - (pirmas_sekm + pd.Timedelta(days=1))).days // 7

print(f"Konfig: mokom {TRAIN_START.date()} -> {TRAIN_END.date()}, blind {BLIND_START.date()} -> {BLIND_END.date()}")

#nutrinam senus parametrus kad modelis is naujo apsimokytu (bet zurnala paliekam)
if os.path.exists(PARAMS_PATH):
    os.remove(PARAMS_PATH)
    print(f'Istrintas: {PARAMS_PATH}')


In [ ]:
#duomenys ir pozymiu inzinerija

#nuskaitom csv ir uzpildom visas dienas (kad nebutu spragu)
df_raw = pd.read_csv(CSV_PATH)
df_raw["Data"] = pd.to_datetime(df_raw["Data"], dayfirst=True, errors="coerce")
df_raw = df_raw.dropna(subset=["Data"]).sort_values("Data").set_index("Data")

paskutine_CSV = df_raw.index.max()
pilnas_indeksas = pd.date_range(TRAIN_START, paskutine_CSV, freq="D")
df = df_raw.reindex(pilnas_indeksas)
df.index.name = "Data"
print(f"Duomenys: {TRAIN_START.date()} -> {paskutine_CSV.date()} ({len(df)} d., truksta: {int(df['Skambuciai'].isna().sum())})")

#trukstamas dienas uzpildom: maza spraga -> linijine interpoliacija, didesne -> ankstesniu metu mediana, blogiausiu atveju 7d vidurkis
df["Skambuciai"] = df["Skambuciai"].interpolate(method="linear", limit=3, limit_area="inside")
for dt in df.index[df["Skambuciai"].isna()]:
    kand = []
    for y in range(TRAIN_START.year, dt.year):
        past_date = dt - pd.DateOffset(years=(dt.year - y))
        if past_date in df.index and not pd.isna(df.at[past_date, "Skambuciai"]):
            kand.append(float(df.at[past_date, "Skambuciai"]))
    if kand:
        df.at[dt, "Skambuciai"] = float(np.median(kand))
df["Skambuciai"] = df["Skambuciai"].fillna(df["Skambuciai"].shift(1).rolling(7, min_periods=1).mean())

#anomaliju paieska tik mokymo lange (pries blind testa)
tm = df.loc[(df.index >= TRAIN_START) & (df.index <= TRAIN_END), "Skambuciai"]
sv = tm.shift(1).rolling(30, min_periods=7).mean()
ss = tm.shift(1).rolling(30, min_periods=7).std()
anomalijos = (((tm - sv).abs() > 3 * ss) & ss.notna()).sum()
print(f"Anomaliju kandidatai (>3 sigma): {int(anomalijos)}")

#LT sventes
LT_SVENTES = pd.to_datetime([
    "2020-01-01","2021-01-01","2022-01-01","2023-01-01","2024-01-01","2025-01-01","2026-01-01",
    "2020-02-16","2021-02-16","2022-02-16","2023-02-16","2024-02-16","2025-02-16","2026-02-16",
    "2020-03-11","2021-03-11","2022-03-11","2023-03-11","2024-03-11","2025-03-11","2026-03-11",
    "2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05",
    "2020-04-13","2021-04-05","2022-04-18","2023-04-10","2024-04-01","2025-04-21","2026-04-06",
    "2020-05-01","2021-05-01","2022-05-01","2023-05-01","2024-05-01","2025-05-01","2026-05-01",
    "2020-06-24","2021-06-24","2022-06-24","2023-06-24","2024-06-24","2025-06-24","2026-06-24",
    "2020-07-06","2021-07-06","2022-07-06","2023-07-06","2024-07-06","2025-07-06","2026-07-06",
    "2020-08-15","2021-08-15","2022-08-15","2023-08-15","2024-08-15","2025-08-15","2026-08-15",
    "2020-11-01","2021-11-01","2022-11-01","2023-11-01","2024-11-01","2025-11-01","2026-11-01",
    "2020-11-02","2021-11-02","2022-11-02","2023-11-02","2024-11-02","2025-11-02","2026-11-02",
    "2020-12-24","2021-12-24","2022-12-24","2023-12-24","2024-12-24","2025-12-24","2026-12-24",
    "2020-12-25","2021-12-25","2022-12-25","2023-12-25","2024-12-25","2025-12-25","2026-12-25",
    "2020-12-26","2021-12-26","2022-12-26","2023-12-26","2024-12-26","2025-12-26","2026-12-26",
])
VELYKOS  = pd.to_datetime(["2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05"])
SEKMINES = pd.to_datetime(["2020-05-31","2021-05-23","2022-06-05","2023-05-28","2024-05-19","2025-06-08","2026-05-24"])
SVENT_SET = set(LT_SVENTES); VEL_SET = set(VELYKOS); SEK_SET = set(SEKMINES)

#funkcija kuri is laiko eilutes padaro lentele su visais pozymiais
def sukurti_pozymius(series: pd.DataFrame) -> pd.DataFrame:
    d = series.copy()
    idx = pd.Series(d.index, index=d.index)
    #kalendoriniai
    dow = d.index.dayofweek
    mon = d.index.month
    d["week_iso"]  = d.index.isocalendar().week.astype(int)
    d["day"]       = d.index.day
    d["quarter"]   = d.index.quarter
    d["dayofyear"] = d.index.dayofyear
    d["is_weekend"] = (dow >= 5).astype(int)
    #sventes
    d["is_holiday"]         = idx.isin(SVENT_SET).astype(int).values
    d["day_before_holiday"] = (idx + pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
    d["day_after_holiday"]  = (idx - pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
    d["is_easter"]          = idx.isin(VEL_SET).astype(int).values
    d["is_pentecost"]       = idx.isin(SEK_SET).astype(int).values
    #ciklinis kodavimas (sin/cos kad pir ir sek butu kaimynai)
    d["dow_sin"] = np.sin(2*np.pi*dow/7); d["dow_cos"] = np.cos(2*np.pi*dow/7)
    d["moy_sin"] = np.sin(2*np.pi*mon/12); d["moy_cos"] = np.cos(2*np.pi*mon/12)
    #lag pozymiai
    for lag in LAGS:
        d[f"lag_{lag}"] = d["Skambuciai"].shift(lag)
    #slenkanti statistika - tik is praeities (shift 1)
    s1 = d["Skambuciai"].shift(1)
    d["roll_mean_7"]  = s1.rolling(7).mean()
    d["roll_mean_14"] = s1.rolling(14).mean()
    d["roll_mean_30"] = s1.rolling(30).mean()
    d["roll_std_7"]   = s1.rolling(7).std()
    d["roll_med_7"]   = s1.rolling(7).median()
    d["roll_min_7"]   = s1.rolling(7).min()
    d["roll_max_7"]   = s1.rolling(7).max()
    return d

df_feat = sukurti_pozymius(df[["Skambuciai"]])

FEATURE_COLS = [c for c in df_feat.columns if c != "Skambuciai"]
print(f"Pozymiu skaicius: {len(FEATURE_COLS)}")
print(f"Pirma diena su visais lag: {df_feat.dropna(subset=[f'lag_{l}' for l in LAGS]).index.min().date()}")


In [ ]:
#padalijimas ir baziniu modeliu metrikos

#efektyvus mokymo rinkinys (be tu eiluciu kur dar nera lag 364)
D_eff = df_feat[(df_feat.index >= EFF_START) & (df_feat.index <= TRAIN_END)] \
           .dropna(subset=[f"lag_{l}" for l in LAGS])
print(f"D_eff: {D_eff.index.min().date()} -> {D_eff.index.max().date()} ({len(D_eff)} d.)")

#2026 m faktai jei jau yra csv
faktai_2026 = df_feat[(df_feat.index >= BLIND_START) &
                      (df_feat.index <= BLIND_END) &
                      df_feat["Skambuciai"].notna()]
if len(faktai_2026):
    print(f"2026 m faktai: {len(faktai_2026)} d. (iki {faktai_2026.index.max().date()})")
else:
    print("2026 m faktu dar nera.")

#baziniai modeliai - kaip atskaitos taskas
if len(faktai_2026):
    y_true = faktai_2026["Skambuciai"].astype(float).values
    hist   = D_eff["Skambuciai"].astype(float).values
    baziniai = {
        "Naive (t-1)":            faktai_2026["lag_1"].values,
        "Seasonal naive (t-7)":   faktai_2026["lag_7"].values,
        "7d slenk. vidurkis":     faktai_2026["roll_mean_7"].values,
        "Same week last year":    faktai_2026["lag_364"].values,
    }
    print("\n=== Baziniai modeliai 2026 m dalyje ===")
    print(f'{"Modelis":26s} {"MAE":>8s} {"RMSE":>8s} {"MAPE%":>8s} {"MASE":>8s}')
    for pav, yp in baziniai.items():
        print(f"{pav:26s} {mae(y_true,yp):8.1f} {rmse(y_true,yp):8.1f} "
              f"{mape(y_true,yp):8.2f} {mase(y_true,yp,hist):8.2f}")
else:
    print("Bazinius skaiciuosim kai bus 2026 faktu.")


In [ ]:
#palyginimas: kuri y transformacija geriausia (raw, log1p, boxcox, poisson)

import xgboost as xgb
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.model_selection import TimeSeriesSplit

X_hpo_tf = D_eff[FEATURE_COLS].values
y_hpo_tf = D_eff["Skambuciai"].values.astype(float)
tscv_tf = TimeSeriesSplit(n_splits=5)

def evaluate_transform(transform_name, y_raw, tscv, X, base_params=None):
    if base_params is None:
        base_params = {
            "n_estimators": 300, "max_depth": 6, "learning_rate": 0.05,
            "subsample": 0.8, "colsample_bytree": 0.8,
            "tree_method": "hist", "random_state": SEED, "n_jobs": -1,
        }
    fold_maes, fold_rmses, fold_mapes = [], [], []
    lam = None

    for tr_i, va_i in tscv.split(X):
        y_tr_raw, y_va_raw = y_raw[tr_i], y_raw[va_i]
        X_tr, X_va = X[tr_i], X[va_i]

        if transform_name == "raw":
            y_tr_t = y_tr_raw
            obj = "reg:squarederror"
        elif transform_name == "log1p":
            y_tr_t = np.log1p(y_tr_raw)
            obj = "reg:squarederror"
        elif transform_name == "boxcox":
            y_tr_shifted = y_tr_raw + 1  #boxcox reikalauja teigiamu
            y_tr_t, lam = boxcox(y_tr_shifted)
            obj = "reg:squarederror"
        elif transform_name == "poisson":
            y_tr_t = y_tr_raw
            obj = "count:poisson"
        else:
            raise ValueError(f"Nezinoma: {transform_name}")

        params = {**base_params, "objective": obj}
        m = xgb.XGBRegressor(**params)
        m.fit(X_tr, y_tr_t, verbose=False)

        pred_t = m.predict(X_va)

        #atvirkstine transformacija
        if transform_name == "log1p":
            pred = np.expm1(pred_t)
        elif transform_name == "boxcox":
            pred = inv_boxcox(pred_t, lam) - 1
        else:
            pred = pred_t

        pred = np.maximum(pred, 0)  #neigiamu nereikia
        fold_maes.append(mae(y_va_raw, pred))
        fold_rmses.append(rmse(y_va_raw, pred))
        fold_mapes.append(mape(y_va_raw, pred))

    return {"Transformacija": transform_name, "MAE": np.mean(fold_maes),
            "RMSE": np.mean(fold_rmses), "MAPE%": np.mean(fold_mapes)}

print("Lyginam y transformacijas (5-fold timeseriessplit)...")
results_tf = []
for tf_name in ["raw", "log1p", "boxcox", "poisson"]:
    try:
        r = evaluate_transform(tf_name, y_hpo_tf, tscv_tf, X_hpo_tf)
        results_tf.append(r)
        print(f"  {tf_name:10s}  MAE={r['MAE']:8.1f}  RMSE={r['RMSE']:8.1f}  MAPE={r['MAPE%']:6.2f}%")
    except Exception as e:
        print(f"  {tf_name:10s}  klaida: {e}")

df_tf = pd.DataFrame(results_tf)

if not df_tf.empty:
    print("\n=== Transformaciju lentele ===")
    print(df_tf.to_string(index=False))
    best_tf = df_tf.loc[df_tf["MAE"].idxmin(), "Transformacija"]
    print(f"\nGeriausia transformacija pagal MAE: {best_tf}")
else:
    print("Visos transformacijos nulauze - tikrint klaidas auksciau.")


In [ ]:
#hiperparametru paieska su optuna - vyksta vieną karta, paskui yra issaugoma

import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit

if os.path.exists(PARAMS_PATH):
    with open(PARAMS_PATH, "r", encoding="utf-8") as f:
        best_params = json.load(f)
    print(f"Hiperparametrai pakrauti is {PARAMS_PATH}")
else:
    print("Paleidziam optuna paieska (100 bandymu, timeseriessplit=5)...")
    try:
        import optuna
        from optuna.samplers import TPESampler
    except ImportError as e:
        raise RuntimeError("Reikalinga optuna: pip install optuna") from e
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    X_hpo = D_eff[FEATURE_COLS].values
    y_hpo_raw = D_eff["Skambuciai"].values.astype(float)

    #imame transformacija is praeito bloko
    try:
        TRANSFORMACIJA = best_tf
    except NameError:
        TRANSFORMACIJA = 'log1p'
    bc_lambda = None

    if TRANSFORMACIJA == "log1p":
        y_hpo = np.log1p(y_hpo_raw)
    elif TRANSFORMACIJA == "boxcox":
        from scipy import stats
        y_hpo, bc_lambda = stats.boxcox(y_hpo_raw + 1)
    else:
        y_hpo = y_hpo_raw

    tscv  = TimeSeriesSplit(n_splits=5)

    def objective(trial):
        if TRANSFORMACIJA in ['log1p', 'boxcox']:
            obj_choices = ['reg:squarederror']
        else:
            obj_choices = ['reg:squarederror', 'count:poisson']
        params = {
            "objective":        trial.suggest_categorical("objective", obj_choices),
            "n_estimators":     trial.suggest_categorical("n_estimators", [100, 300, 500, 1000]),
            "max_depth":        trial.suggest_int("max_depth", 3, 9),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":        trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 0.1, 10.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
            "gamma":            trial.suggest_float("gamma", 0.0, 0.3),
            "tree_method":      "hist",
            "random_state":     SEED,
            "n_jobs":           -1,
        }
        fold_mases = []
        for tr, va in tscv.split(X_hpo):
            m = xgb.XGBRegressor(**params, early_stopping_rounds=30)
            m.fit(X_hpo[tr], y_hpo[tr], eval_set=[(X_hpo[va], y_hpo[va])], verbose=False)
            p_tf = m.predict(X_hpo[va])

            if TRANSFORMACIJA == "log1p":
                p = np.expm1(p_tf)
            elif TRANSFORMACIJA == "boxcox":
                from scipy.special import inv_boxcox
                p = inv_boxcox(p_tf, bc_lambda) - 1
            else:
                p = p_tf

            p = np.maximum(0, p)

            #klaida vertinama originalia skale
            y_tr_raw = y_hpo_raw[tr]
            y_va_raw = y_hpo_raw[va]

            denom = float(np.mean(np.abs(y_tr_raw[7:] - y_tr_raw[:-7])))
            denom = denom if denom > 0 else 1.0

            fold_mases.append(np.mean(np.abs(y_va_raw - p)) / denom)

        return float(np.mean(fold_mases))

    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
    study.optimize(objective, n_trials=100, show_progress_bar=False)
    best_params = study.best_params
    best_params.update({"objective": study.best_params.get("objective", "reg:squarederror"),
                        "tree_method": "hist", "random_state": SEED})
    with open(PARAMS_PATH, "w", encoding="utf-8") as f:
        json.dump(best_params, f, indent=2)
    print(f"Paieska baigta. MASE(val)={study.best_value:.3f}. Issaugot: {PARAMS_PATH}")

print("Hiperparametrai:")
for k, v in best_params.items():
    print(f"  {k:18s} {v}")


In [ ]:
#mokomas modelis + prognozuojam ateinancia savaite + zurnalas

#mokymo aibė iki paskutinės turimos dienos
best_params.pop("_train_end", None)
mokymo = df_feat[(df_feat.index >= EFF_START) &
                 df_feat["Skambuciai"].notna()] \
             .dropna(subset=[f"lag_{l}" for l in LAGS])
paskutine_turima = mokymo.index.max()
print(f"Mokymo rinkinys: {mokymo.index.min().date()} -> {paskutine_turima.date()} ({len(mokymo)} d.)")

X_tr = mokymo[FEATURE_COLS].values
y_tr_raw = mokymo["Skambuciai"].values.astype(float)

#transformacija
try:
    TRANSFORMACIJA = best_tf
except NameError:
    TRANSFORMACIJA = "log1p"
bc_lambda = None

if TRANSFORMACIJA == "log1p":
    y_tr = np.log1p(y_tr_raw)
elif TRANSFORMACIJA == "boxcox":
    from scipy import stats
    y_tr, bc_lambda = stats.boxcox(y_tr_raw + 1)
else:
    y_tr = y_tr_raw

modelis = xgb.XGBRegressor(**best_params)
modelis.fit(X_tr, y_tr)
print(f"XGBoost apmokytas (transformacija: {TRANSFORMACIJA}).")

def kita_savaite(paskutine: pd.Timestamp):
    #grazina pir-sek savaite (arba galo gabalą)
    start = paskutine + pd.Timedelta(days=1)
    if start > BLIND_END:
        return None
    days_until_sunday = 6 - start.dayofweek
    week_end = start + pd.Timedelta(days=days_until_sunday)
    week_end = min(week_end, BLIND_END)
    return pd.date_range(start, week_end, freq="D")

def prognozuoti_savaite(model, istorija: pd.DataFrame, savaite, transformacija="log1p", lam=None):
    #rekursyvi prognoze - po kiekvienos dienos atnaujinam lag
    d = istorija[["Skambuciai"]].copy()
    trukstamos = [dt for dt in savaite if dt not in d.index]
    if trukstamos:
        d = pd.concat([d, pd.DataFrame({"Skambuciai": np.nan}, index=trukstamos)]).sort_index()
    prognozes = {}
    for dt in savaite:
        feat = sukurti_pozymius(d)
        row = feat.loc[[dt], FEATURE_COLS].fillna(0.0).values
        y_pred_tf = float(model.predict(row)[0])

        if transformacija == "log1p":
            y_pred = np.expm1(y_pred_tf)
        elif transformacija == "boxcox":
            from scipy.special import inv_boxcox
            y_pred = inv_boxcox(y_pred_tf, lam) - 1
        else:
            y_pred = y_pred_tf

        y_pred = max(0, y_pred)
        prognozes[dt] = y_pred
        d.at[dt, "Skambuciai"] = y_pred  #savo prognoze panaudosim toliau
    return pd.Series(prognozes, name="Prognoze")

#zurnalo nuskaitymas (saugomas nuolat)
STULP = ["Savaites_Nr", "Prognoze", "CI_Lower", "CI_Upper", "Faktas", "Abs_Klaida", "Proc_Klaida"]
if os.path.exists(LOG_PATH):
    zurnalas = pd.read_csv(LOG_PATH, index_col="Data", parse_dates=True)
    truksta = [c for c in STULP if c not in zurnalas.columns]
    for c in truksta:
        zurnalas[c] = np.nan
    zurnalas = zurnalas[STULP]
else:
    zurnalas = pd.DataFrame(columns=STULP)
    zurnalas.index = pd.DatetimeIndex([], name="Data")
print(f"Zurnale: {len(zurnalas)} irasu")

#uzpildom zurnala faktais
uzpildyta = 0
for dt in zurnalas.index:
    if pd.isna(zurnalas.at[dt, "Faktas"]) and dt in df_feat.index \
            and not pd.isna(df_feat.at[dt, "Skambuciai"]):
        f = float(df_feat.at[dt, "Skambuciai"])
        p = float(zurnalas.at[dt, "Prognoze"])
        zurnalas.at[dt, "Faktas"]      = f
        zurnalas.at[dt, "Abs_Klaida"]  = abs(f - p)
        zurnalas.at[dt, "Proc_Klaida"] = abs(f - p) / f * 100 if f != 0 else np.nan
        uzpildyta += 1
if uzpildyta:
    print(f"Uzpildyta {uzpildyta} faktu zurnale")

#paskutines savaites paklaida
uzbaigta = zurnalas.dropna(subset=["Faktas"])
if len(uzbaigta):
    pask_nr = int(uzbaigta["Savaites_Nr"].max())
    grp = uzbaigta[uzbaigta["Savaites_Nr"] == pask_nr]
    if len(grp) == 7:
        y_f = grp["Faktas"].values.astype(float)
        y_p = grp["Prognoze"].values.astype(float)
        print(f"\n--- Praejusi savaite {pask_nr}: {grp.index.min().date()} -> {grp.index.max().date()} ---")
        print(f"  MAE  : {mae(y_f, y_p):8.1f}")
        print(f"  RMSE : {rmse(y_f, y_p):8.1f}")
        print(f"  MAPE : {mape(y_f, y_p):8.2f}%")
        print(f"  MASE : {mase(y_f, y_p, D_eff['Skambuciai'].values):8.2f}")

#nauja prognoze ateinanciai savaitei
savaite = kita_savaite(paskutine_turima)
if savaite is None:
    print("\n2026 m prognozavimas baigtas (pasieke 2026-03-31).")
else:
    pirm, sekm = savaite[0], savaite[-1]
    sav_nr = savaites_nr_metuose(pirm)
    print(f"\nProgostozuojam savaite {sav_nr} ({pirm.year}): {pirm.date()} -> {sekm.date()}")

    prog = prognozuoti_savaite(modelis, df_feat, savaite, TRANSFORMACIJA, bc_lambda)

    #sukuriam ateities pozymius su gauta prognoze (kad bootstrap veiktu)
    temp_df = df_feat[["Skambuciai"]].copy()
    future_rows = pd.DataFrame({"Skambuciai": prog.values}, index=savaite)
    temp_df = pd.concat([temp_df, future_rows])
    temp_feat = sukurti_pozymius(temp_df)
    X_fut = temp_feat.loc[savaite, FEATURE_COLS].fillna(0.0).values

    #bootstrap pasikliautiniam intervalui
    print(f"Generuojam 90% pasikliautinaji interval (100 bootstrap iteraciju)...")
    N_BOOT = 100
    ALPHA_PI = 0.10
    boot_preds = np.zeros((N_BOOT, len(savaite)))
    for b in range(N_BOOT):
        rng = np.random.RandomState(SEED + b)
        idx = rng.choice(len(X_tr), size=len(X_tr), replace=True)
        bp = {**best_params, "random_state": SEED + b}
        bp.pop("_train_end", None)
        m_b = xgb.XGBRegressor(**bp)
        m_b.fit(X_tr[idx], y_tr[idx])
        pred_tf = m_b.predict(X_fut)
        if TRANSFORMACIJA == "log1p":
            pred_y = np.expm1(pred_tf)
        elif TRANSFORMACIJA == "boxcox":
            from scipy.special import inv_boxcox
            pred_y = inv_boxcox(pred_tf, bc_lambda) - 1
        else:
            pred_y = pred_tf
        boot_preds[b] = pred_y

    lower_fut = np.percentile(boot_preds, 100 * ALPHA_PI / 2, axis=0)
    upper_fut = np.percentile(boot_preds, 100 * (1 - ALPHA_PI / 2), axis=0)

    for i, dt in enumerate(savaite):
        zurnalas.loc[dt] = {
            "Savaites_Nr": sav_nr,
            "Prognoze":    int(round(prog[dt])),
            "CI_Lower":    max(0, int(round(lower_fut[i]))),
            "CI_Upper":    max(0, int(round(upper_fut[i]))),
            "Faktas":      np.nan,
            "Abs_Klaida":  np.nan,
            "Proc_Klaida": np.nan,
        }
    print("\nDienos prognozes:")
    for i, dt in enumerate(savaite):
        diena = ["Pir","Ant","Tre","Ket","Pen","Ses","Sek"][dt.dayofweek]
        p = int(round(prog[dt]))
        lo = max(0, int(round(lower_fut[i])))
        hi = max(0, int(round(upper_fut[i])))
        print(f"  {dt.date()} ({diena}): {p:5d}   [90% CI: {lo:5d} - {hi:5d}]")

zurnalas.index.name = "Data"
zurnalas.sort_index(inplace=True)
zurnalas.to_csv(LOG_PATH)
print(f"\nZurnalas issaugotas: {LOG_PATH}")


In [ ]:
#grafika, shap, suvestine

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

#grafikai 2026 q1
dates_2026 = zurnalas[zurnalas.index >= "2026-01-01"].index

if len(dates_2026) > 0:
    #ateities lag pozymiams
    d_temp = df[["Skambuciai"]].copy()
    trukstamos = [dt for dt in dates_2026 if dt not in d_temp.index]
    if trukstamos:
        d_temp = pd.concat([d_temp, pd.DataFrame({"Skambuciai": np.nan}, index=trukstamos)]).sort_index()
    for dt in dates_2026:
        if pd.isna(d_temp.at[dt, "Skambuciai"]):
            d_temp.at[dt, "Skambuciai"] = zurnalas.at[dt, "Prognoze"]
    feat_temp = sukurti_pozymius(d_temp)
    df_2026_q1 = feat_temp.loc[dates_2026].copy()

    xgb_pred = zurnalas.loc[dates_2026, "Prognoze"].values
    ci_lo = zurnalas.loc[dates_2026, "CI_Lower"].values.astype(float)
    ci_hi = zurnalas.loc[dates_2026, "CI_Upper"].values.astype(float)
    faktas_reiksmes = zurnalas.loc[dates_2026, "Faktas"].values
    atsarginis_vidurkis = df["Skambuciai"].mean()

    baseline_vals = {
        "Naive (t-1)":      df_2026_q1["lag_1"].fillna(atsarginis_vidurkis).values,
        "Seasonal (t-7)":   df_2026_q1["lag_7"].fillna(atsarginis_vidurkis).values,
        "SameYear (t-364)": df_2026_q1["lag_364"].fillna(atsarginis_vidurkis).values,
    }

    fig, axes = plt.subplots(2, 1, figsize=(15, 10))

    #pirmas grafikas - prognoze vs faktas
    ax = axes[0]
    colors = ["orange", "green", "red"]
    for (name, vals), color in zip(baseline_vals.items(), colors):
        ax.plot(dates_2026, vals, label=name, linewidth=1.5, alpha=0.6, color=color, linestyle="--")

    ax.fill_between(dates_2026, ci_lo, ci_hi, color="royalblue", alpha=0.18, label="XGBoost 90% CI")
    ax.plot(dates_2026, xgb_pred, label="XGBoost", linewidth=2.5, color="royalblue", marker="o", markersize=4, zorder=3)

    if not pd.isna(faktas_reiksmes).all():
        ax.plot(dates_2026, faktas_reiksmes, label="Faktas", linewidth=3, color="black", marker="s", markersize=5, zorder=5)

    ax.set_xlabel("Data (2026)", fontsize=11)
    ax.set_ylabel("Skambuciu kiekis", fontsize=11)
    ax.set_title(f"XGBoost vs baseline vs faktas (iki {dates_2026.max().date()})", fontsize=13, fontweight="bold")
    ax.legend(loc="best", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_minor_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

    #antras grafikas - kaupiamoji suma
    ax = axes[1]
    xgb_cumsum = np.cumsum(xgb_pred)
    for (name, vals), color in zip(baseline_vals.items(), colors):
        cumsum = np.cumsum(vals)
        ax.plot(dates_2026, cumsum, label=f"{name} (kumul.)", linewidth=1.5, alpha=0.6, color=color, linestyle="--")

    ax.plot(dates_2026, xgb_cumsum, label="XGBoost (kumul.)", linewidth=2.5, color="royalblue", marker="o", markersize=4, zorder=3)

    if not pd.isna(faktas_reiksmes).all():
        faktas_cumsum = pd.Series(faktas_reiksmes).cumsum()
        ax.plot(dates_2026, faktas_cumsum, label="Faktas (kumul.)", linewidth=3, color="black", marker="s", markersize=5, zorder=5)

    ax.set_xlabel("Data (2026)", fontsize=11)
    ax.set_ylabel("Is viso skambuciu", fontsize=11)
    ax.set_title(f"Kaupiamoji suma (iki {dates_2026.max().date()})", fontsize=13, fontweight="bold")
    ax.legend(loc="best", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_minor_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

    #suvestine
    print("\n" + "="*70)
    print(f"SUVESTINE (iki {dates_2026.max().date()})")
    print("="*70)
    print(f"\n{'Modelis':<25} {'Vidut./d':<15} {'Viso':<15} {'Min':<10} {'Max':<10}")
    print("-"*70)

    if not pd.isna(faktas_reiksmes).all():
        valid_faktas = faktas_reiksmes[~pd.isna(faktas_reiksmes)]
        print(f"{'FAKTAS':<25} {valid_faktas.mean():>12.0f}  {valid_faktas.sum():>12.0f}  {valid_faktas.min():>8.0f}  {valid_faktas.max():>8.0f}")
        print("-" * 70)

    print(f"{'XGBoost':<25} {xgb_pred.mean():>12.0f}  {xgb_cumsum[-1]:>12.0f}  {xgb_pred.min():>8.0f}  {xgb_pred.max():>8.0f}")

    for name, vals in baseline_vals.items():
        cumsum = np.cumsum(vals)
        print(f"{name:<25} {vals.mean():>12.0f}  {cumsum[-1]:>12.0f}  {vals.min():>8.0f}  {vals.max():>8.0f}")
    print("="*70)

else:
    print("Zurnale dar nera 2026 prognoziu, grafiku nera.")


#bendros 2026 metrikos
uzbaigta = zurnalas.dropna(subset=["Faktas"]).copy()
if len(uzbaigta) == 0:
    print("\n2026 faktu dar nera.")
else:
    y_f = uzbaigta["Faktas"].values.astype(float)
    y_p = uzbaigta["Prognoze"].values.astype(float)
    hist = D_eff["Skambuciai"].values.astype(float)
    print(f"\n=== 2026 m bendros metrikos ({len(uzbaigta)} d.) ===")
    print(f"  MAE  : {mae(y_f, y_p):8.2f}")
    print(f"  RMSE : {rmse(y_f, y_p):8.2f}")
    print(f"  MAPE : {mape(y_f, y_p):8.2f}%")
    print(f"  MASE : {mase(y_f, y_p, hist):8.2f}")

#shap pozymiu svarba
try:
    import shap
    X_sh = mokymo[FEATURE_COLS]
    explainer = shap.TreeExplainer(modelis)
    shap_values = explainer.shap_values(X_sh)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sh, show=False, max_display=15)
    plt.title("SHAP pozymiu svarba (XGBoost)", fontsize=12, pad=12)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("shap neidiegtas (pip install shap). Praleidziam.")

#adaptacija - mape per savaites
if len(uzbaigta) >= 7:
    uzbaigta["Proc_Klaida"] = np.abs(uzbaigta["Faktas"] - uzbaigta["Prognoze"]) / uzbaigta["Faktas"] * 100
    sav_mape = uzbaigta.groupby("Savaites_Nr")["Proc_Klaida"].mean()
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(sav_mape.index.astype(int), sav_mape.values, "o-", color="crimson", linewidth=2, label="Savaites MAPE")
    ax.axhline(10, color="green", linestyle="--", label="10% riba")
    ax.set_xlabel("2026 m savaites nr.")
    ax.set_ylabel("MAPE (%)")
    ax.set_title("XGBoost adaptacija po reformos")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("\nReikia bent 1 pilnos savaites adaptacijos grafikui.")


In [ ]:
#papildomos metrikos: smape, r2, likuciai, ljung-box

import numpy as np

def smape(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    denom = (np.abs(y) + np.abs(yhat)) / 2.0
    mask = denom != 0
    return float(np.mean(np.abs(y[mask] - yhat[mask]) / denom[mask]) * 100) if mask.any() else np.nan

def r2_score_manual(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

uzbaigta_6 = zurnalas.dropna(subset=["Faktas"]).copy()
if len(uzbaigta_6) > 0:
    y_f = uzbaigta_6["Faktas"].values.astype(float)
    y_p = uzbaigta_6["Prognoze"].values.astype(float)
    print(f"=== 2026 m papildomos metrikos ({len(uzbaigta_6)} d.) ===")
    print(f"  sMAPE : {smape(y_f, y_p):8.2f}%")
    print(f"  R2    : {r2_score_manual(y_f, y_p):8.4f}")
else:
    print("2026 faktu dar nera.")

#likuciai mokymo aibeje (in-sample)
y_hat_train = modelis.predict(mokymo[FEATURE_COLS].values)
y_obs_train = mokymo["Skambuciai"].values.astype(float)
resid_train = y_obs_train - y_hat_train

print(f"\nMokymo likuciai:")
print(f"  vidurkis : {resid_train.mean():8.2f}")
print(f"  std      : {resid_train.std():8.2f}")
print(f"  min/max  : {resid_train.min():8.1f} / {resid_train.max():8.1f}")

#acf grafikas
try:
    from statsmodels.graphics.tsaplots import plot_acf
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(mokymo.index, resid_train, color="royalblue", linewidth=0.6)
    axes[0].axhline(0, color="black", linewidth=0.5)
    axes[0].set_title("XGBoost mokymo likuciai")
    axes[0].grid(True, alpha=0.3)
    plot_acf(resid_train, lags=40, ax=axes[1])
    axes[1].set_title("Likuciu ACF (40 lagu)")
    plt.tight_layout(); plt.show()
except ImportError:
    print("statsmodels nerastas, acf praleistas.")

#ljung-box testas
try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    lb = acorr_ljungbox(resid_train, lags=[7, 14, 28], return_df=True)
    print("\nLjung-Box testas:")
    for lag, row in lb.iterrows():
        verdict = "baltas triuksmas" if row["lb_pvalue"] > 0.05 else "yra autokoreliaciju"
        print(f"  lag={int(lag):3d}: Q={row['lb_stat']:8.2f}  p={row['lb_pvalue']:.4f}  -> {verdict}")
except ImportError:
    print("statsmodels nerastas, ljung-box praleistas.")


In [ ]:
#dm-hln testas: ar xgboost statistiskai geresnis nei baziniai modeliai

def dm_test(e1, e2, h=1, power=2):
    #diebold-mariano su hln korekcija
    from scipy import stats
    d = np.abs(e1)**power - np.abs(e2)**power
    T = len(d)
    d_bar = np.mean(d)
    gamma = np.array([np.sum((d[i:] - d_bar) * (d[:T-i] - d_bar)) / T for i in range(h)])
    V = (gamma[0] + 2 * np.sum(gamma[1:])) / T
    if V <= 0:
        return np.nan, np.nan
    DM_stat = d_bar / np.sqrt(V)
    hln_corr = np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    DM_hln = DM_stat * hln_corr
    p_value = 2 * (1 - stats.t.cdf(abs(DM_hln), df=T-1))
    return DM_hln, p_value

def holm_bonferroni(p_values, alpha=0.05):
    m = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p = np.array(p_values)[sorted_idx]
    results = []
    rejected_any = True
    for i, (idx, p) in enumerate(zip(sorted_idx, sorted_p)):
        adj_alpha = alpha / (m - i)
        reject = p < adj_alpha and rejected_any
        if not reject:
            rejected_any = False
        results.append((idx, p, adj_alpha, reject))
    return results

if len(faktai_2026):
    y_true_dm = faktai_2026["Skambuciai"].astype(float).values
    zurnalas_2026 = zurnalas.loc[faktai_2026.index].dropna(subset=["Faktas", "Prognoze"])
    if len(zurnalas_2026) >= 7:
        y_true_dm = zurnalas_2026["Faktas"].values.astype(float)
        y_xgb_dm = zurnalas_2026["Prognoze"].values.astype(float)
        e_xgb = y_true_dm - y_xgb_dm

        baziniai_dm = {
            "Naive (t-1)": faktai_2026.loc[zurnalas_2026.index, "lag_1"].values,
            "Seasonal (t-7)": faktai_2026.loc[zurnalas_2026.index, "lag_7"].values,
            "7d vidurkis": faktai_2026.loc[zurnalas_2026.index, "roll_mean_7"].values,
        }

        print("=== DM-HLN testas: XGBoost vs baziniai ===")
        p_values = []
        names = []
        for name, y_baz in baziniai_dm.items():
            e_baz = y_true_dm - y_baz
            dm_stat, p_val = dm_test(e_xgb, e_baz, h=7, power=2)
            p_values.append(p_val)
            names.append(name)
            verdict = "XGBoost geriau" if (p_val < 0.05 and dm_stat < 0) else \
                      "Bazinis geriau" if (p_val < 0.05 and dm_stat > 0) else \
                      "nera reiksmingo skirtumo"
            print(f"  XGBoost vs {name:20s}: DM={dm_stat:7.3f}  p={p_val:.4f}  -> {verdict}")

        hb_results = holm_bonferroni(p_values)
        print("\n  Holm-Bonferroni korekcija:")
        for idx, p, adj_a, reject in hb_results:
            status = "ATMESTA" if reject else "NEATMESTA"
            print(f"    {names[idx]:20s}: p={p:.4f}  adj_alpha={adj_a:.4f}  -> {status}")
    else:
        print("Per mazai duomenu DM-HLN testui (reikia >=7 d).")
else:
    print("2026 faktu dar nera, dm-hln praleistas.")


In [ ]:
#wilcoxon ir mann-kendall: ar klaidos stabilios laike 

from scipy import stats

def mann_kendall_test(x):
    n = len(x)
    s = 0
    for k in range(n - 1):
        for j in range(k + 1, n):
            s += np.sign(x[j] - x[k])
    var_s = n * (n - 1) * (2 * n + 5) / 18
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    trend = "didejantis" if z > 0 else "mazejantis" if z < 0 else "nera trendo"
    return z, p_value, trend

#mokymo likuciu trendai
y_train_pred = modelis.predict(mokymo[FEATURE_COLS].values)
resid_full = mokymo["Skambuciai"].values.astype(float) - y_train_pred
abs_errors = np.abs(resid_full)

print("=== Wilcoxon ir Mann-Kendall  ===")

#mann-kendall - ar absoliucios klaidos turi trendą
weekly_errors = pd.Series(abs_errors, index=mokymo.index).resample('W').mean().dropna()
mk_z, mk_p, mk_trend = mann_kendall_test(weekly_errors.values)
print(f"\nMann-Kendall (savaitines abs klaidos):")
print(f"  Z={mk_z:.4f}  p={mk_p:.4f}  trendas: {mk_trend}")
if mk_p < 0.05:
    print(f"  -> reiksmingas trendas (p<0.05)")
else:
    print(f"  -> nera reiksmingo trendo")

#wilcoxon - pirma vs antra puse
mid = len(abs_errors) // 2
first_half = abs_errors[:mid]
second_half = abs_errors[mid:mid+len(first_half)]
w_stat, w_p = stats.wilcoxon(first_half, second_half)
print(f"\nWilcoxon (1-a vs 2-a mokymo puse):")
print(f"  W={w_stat:.2f}  p={w_p:.4f}")
if w_p < 0.05:
    print(f"  -> reiksmingas skirtumas")
else:
    print(f"  -> nera reiksmingo skirtumo")


In [ ]:
#klaidu analize pagal savaites diena, menesi ir sventes

import matplotlib.pyplot as plt

train_dates = mokymo.index
y_train_actual = mokymo["Skambuciai"].values.astype(float)
y_train_model = modelis.predict(mokymo[FEATURE_COLS].values)
train_errors = y_train_actual - y_train_model
train_abs_errors = np.abs(train_errors)
train_pct_errors = np.where(y_train_actual != 0, np.abs(train_errors) / y_train_actual * 100, np.nan)

df_errors = pd.DataFrame({
    "abs_error": train_abs_errors,
    "pct_error": train_pct_errors,
    "dayofweek": train_dates.dayofweek,
    "month": train_dates.month,
    "is_holiday": [1 if d in SVENT_SET else 0 for d in train_dates],
}, index=train_dates)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

#savaites diena
ax = axes[0]
dow_names = ["Pir", "Ant", "Tre", "Ket", "Pen", "Ses", "Sek"]
dow_errors = df_errors.groupby("dayofweek")["abs_error"].agg(["mean", "std"])
ax.bar(range(7), dow_errors["mean"], yerr=dow_errors["std"], color="royalblue", alpha=0.7, capsize=4)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_title("Vidutine abs klaida pagal savaites diena")
ax.set_ylabel("MAE")
ax.grid(True, alpha=0.3)

#menuo
ax = axes[1]
mon_names = ["Sau", "Vas", "Kov", "Bal", "Geg", "Bir", "Lie", "Rgp", "Rgs", "Spa", "Lap", "Grd"]
mon_errors = df_errors.groupby("month")["abs_error"].agg(["mean", "std"])
ax.bar(range(1, 13), mon_errors["mean"], yerr=mon_errors["std"], color="forestgreen", alpha=0.7, capsize=4)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(mon_names, fontsize=9)
ax.set_title("Vidutine abs klaida pagal menesi")
ax.set_ylabel("MAE")
ax.grid(True, alpha=0.3)

#sventes vs nesventes
ax = axes[2]
hol_groups = df_errors.groupby("is_holiday")["abs_error"].agg(["mean", "std"])
labels = ["Ne sventes", "Sventes"]
ax.bar(labels, hol_groups["mean"], yerr=hol_groups["std"], color=["steelblue", "tomato"], alpha=0.7, capsize=4)
ax.set_title("Sventes vs nesventes")
ax.set_ylabel("MAE")
ax.grid(True, alpha=0.3)

plt.suptitle("XGBoost klaidu analize (in-sample)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n=== Klaidos pagal savaites diena ===")
for dow in range(7):
    subset = df_errors[df_errors["dayofweek"] == dow]
    print(f"  {dow_names[dow]}: MAE={subset['abs_error'].mean():.1f}  MAPE={subset['pct_error'].mean():.2f}%  n={len(subset)}")

print("\n=== Sventes vs nesventes ===")
for hol in [0, 1]:
    subset = df_errors[df_errors["is_holiday"] == hol]
    lbl = "Sventes" if hol else "Ne sventes"
    print(f"  {lbl:12s}: MAE={subset['abs_error'].mean():.1f}  MAPE={subset['pct_error'].mean():.2f}%  n={len(subset)}")


In [ ]:
#bootstrap prognozes intervalai + picp/pinaw/winkler

N_BOOT = 100
ALPHA_PI = 0.10  #90% PI

print(f"Mokom {N_BOOT} bootstrap modeliu...")

X_boot_train = mokymo[FEATURE_COLS].values
y_boot_train = mokymo["Skambuciai"].values.astype(float)

#jei turim 2026 faktu - testavimui imam juos, jei ne - paskutines 90 d
if len(faktai_2026) and len(zurnalas.dropna(subset=["Faktas"])):
    z26 = zurnalas.loc[faktai_2026.index].dropna(subset=["Faktas"])
    X_boot_test = faktai_2026.loc[z26.index, FEATURE_COLS].values
    y_boot_test = z26["Faktas"].values.astype(float)
    boot_test_source = "blind 2026"
else:
    X_boot_test = X_boot_train[-90:]
    y_boot_test = y_boot_train[-90:]
    boot_test_source = "in-sample (paskutines 90 d.)"

boot_predictions = np.zeros((N_BOOT, len(y_boot_test)))

for b in range(N_BOOT):
    rng = np.random.RandomState(SEED + b)
    idx = rng.choice(len(X_boot_train), size=len(X_boot_train), replace=True)
    bp = {**best_params, "random_state": SEED + b}
    m_b = xgb.XGBRegressor(**bp)
    m_b.fit(X_boot_train[idx], y_boot_train[idx])
    boot_predictions[b] = m_b.predict(X_boot_test)

print(f"Bootstrap baigtas. Test saltinis: {boot_test_source}")

lower = np.percentile(boot_predictions, 100 * ALPHA_PI / 2, axis=0)
upper = np.percentile(boot_predictions, 100 * (1 - ALPHA_PI / 2), axis=0)
point_pred = np.mean(boot_predictions, axis=0)

#picp - kiek faktu pateko i interval
in_interval = (y_boot_test >= lower) & (y_boot_test <= upper)
picp = np.mean(in_interval)

#pinaw - normalizuotas vidutinis plotis
y_range = np.max(y_boot_test) - np.min(y_boot_test)
pinaw = np.mean(upper - lower) / y_range if y_range > 0 else np.nan

def winkler_score(y_true, lower, upper, alpha=0.05):
    n = len(y_true)
    score = 0
    for i in range(n):
        width = upper[i] - lower[i]
        if y_true[i] < lower[i]:
            score += width + (2 / alpha) * (lower[i] - y_true[i])
        elif y_true[i] > upper[i]:
            score += width + (2 / alpha) * (y_true[i] - upper[i])
        else:
            score += width
    return score / n

ws = winkler_score(y_boot_test, lower, upper, ALPHA_PI)

print(f"\n=== Bootstrap {int((1-ALPHA_PI)*100)}% PI metrikos ===")
print(f"  PICP   : {picp:.4f}  (norima >= {1-ALPHA_PI:.2f})")
print(f"  PINAW  : {pinaw:.4f}  (maziau = geriau)")
print(f"  Winkler: {ws:.2f}   (maziau = geriau)")

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(y_boot_test))
ax.fill_between(x, lower, upper, alpha=0.25, color="royalblue", label=f"{int((1-ALPHA_PI)*100)}% PI")
ax.plot(x, y_boot_test, "k-", linewidth=1.5, label="Faktas", alpha=0.8)
ax.plot(x, point_pred, "b--", linewidth=1, label="Bootstrap vidurkis", alpha=0.7)
ax.set_xlabel("Testo dienos")
ax.set_ylabel("Skambuciai")
ax.set_title(f"XGBoost bootstrap {int((1-ALPHA_PI)*100)}% PI  "
             f"(PICP={picp:.3f}, PINAW={pinaw:.3f}, Winkler={ws:.1f})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
#priedas a: papildomas vertinimas su train_test_split
#paprastas atsitiktinis padalijimas - ne pagrindinis vertinimas, tik sanity check (loginio nuoseklumo patikra)

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

TEST_SIZE = 0.15     #~15% dienu testavimui
RANDOM_STATE = SEED

X_full = D_eff[FEATURE_COLS].values
y_full_vals = D_eff["Skambuciai"].values.astype(float)
print(f"D_eff: X={X_full.shape}, y={y_full_vals.shape}")

X_trA, X_teA, y_trA, y_teA = train_test_split(
    X_full, y_full_vals, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True,
)
print(f"Train: {len(X_trA)} d.   Test: {len(X_teA)} d.   (test_size={TEST_SIZE})")

#kiek dienu per metus pateko i testa
np.random.seed(RANDOM_STATE)
shuffled_pos = np.arange(len(D_eff))
np.random.shuffle(shuffled_pos)
n_test = int(round(len(D_eff) * TEST_SIZE))
test_positions = shuffled_pos[:n_test]
test_dates = D_eff.index[test_positions]
per_year = pd.Series(test_dates.year).value_counts().sort_index()
print("\nTestu skaicius pagal metus:")
for yr, cnt in per_year.items():
    print(f"  {yr}: {cnt} d.")

modelis_A = xgb.XGBRegressor(**best_params)
modelis_A.fit(X_trA, y_trA)
y_pred_te = modelis_A.predict(X_teA)

maeA  = mean_absolute_error(y_teA, y_pred_te)
rmseA = float(np.sqrt(mean_squared_error(y_teA, y_pred_te)))
y_nz  = y_teA != 0
mapeA = float(np.mean(np.abs((y_teA[y_nz] - y_pred_te[y_nz]) / y_teA[y_nz])) * 100) if y_nz.any() else np.nan
smapeA = smape(y_teA, y_pred_te) if "smape" in dir() else float(np.mean(
    np.abs(y_teA - y_pred_te) / ((np.abs(y_teA) + np.abs(y_pred_te)) / 2.0)) * 100)
r2A   = r2_score(y_teA, y_pred_te)

print("\n=== Priedas A - train_test_split rezultatai (XGBoost) ===")
print(f"  MAE   : {maeA:8.2f}")
print(f"  RMSE  : {rmseA:8.2f}")
print(f"  MAPE  : {mapeA:8.2f}%")
print(f"  sMAPE : {smapeA:8.2f}%")
print(f"  R2    : {r2A:8.4f}")
print("\nPastaba: rezultatai optimistiniai - random padalijimas leidzia modeliui")
print("matyti chronologiskai artimas dienas per lag pozymius.")

#grafikai
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_teA, y_pred_te, alpha=0.4, s=15, color="royalblue", edgecolors="none")
mn, mx = min(y_teA.min(), y_pred_te.min()), max(y_teA.max(), y_pred_te.max())
ax.plot([mn, mx], [mn, mx], "k--", linewidth=1, label="Ideali (y=x)")
ax.set_xlabel("Faktas"); ax.set_ylabel("Prognoze")
ax.set_title(f"XGBoost priedas A: faktas vs prognoze (n={len(y_teA)})")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
sort_idx = np.argsort(test_dates)
dates_sorted = test_dates[sort_idx]
y_f_sorted = y_teA[sort_idx]
y_p_sorted = y_pred_te[sort_idx]
step = max(1, len(dates_sorted) // 80)
ax.plot(range(len(y_f_sorted[::step])), y_f_sorted[::step], "k-", linewidth=1.5, label="Faktas", alpha=0.8)
ax.plot(range(len(y_p_sorted[::step])), y_p_sorted[::step], "b-", linewidth=1.5, label="XGBoost", alpha=0.7)
ax.fill_between(range(len(y_f_sorted[::step])), y_f_sorted[::step], y_p_sorted[::step], alpha=0.15, color="royalblue")
ax.set_xlabel("Testo dienos (chronologine)"); ax.set_ylabel("Skambuciai")
ax.set_title("XGBoost priedas A: chronologine juosta")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
#priedas b: metrikos pagal metus + likuciu diagnostika

print("=== Priedas b - mokymo metrikos pagal metus ===")
y_train_pred_full = modelis.predict(mokymo[FEATURE_COLS].values)
y_train_full = mokymo["Skambuciai"].values.astype(float)
years = mokymo.index.year

for yr in sorted(years.unique()):
    mask = years == yr
    y_yr = y_train_full[mask]
    p_yr = y_train_pred_full[mask]
    print(f"  {yr}: MAE={mae(y_yr, p_yr):8.1f}  RMSE={rmse(y_yr, p_yr):8.1f}  MAPE={mape(y_yr, p_yr):6.2f}%  n={mask.sum()}")

#likucai
residuals = y_train_full - y_train_pred_full

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(mokymo.index, residuals, color="royalblue", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Likuciu laiko eilute")
ax.set_ylabel("Likana")
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.hist(residuals, bins=50, color="royalblue", alpha=0.7, edgecolor="white")
ax.axvline(0, color="red", linewidth=1, linestyle="--")
ax.set_title(f"Histograma (vid={np.mean(residuals):.1f}, std={np.std(residuals):.1f})")
ax.set_xlabel("Likana")
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
from scipy import stats
stats.probplot(residuals, plot=ax)
ax.set_title("Q-Q grafikas")
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
df_resid = pd.DataFrame({"year": years, "residual": residuals})
df_resid.boxplot(column="residual", by="year", ax=ax)
ax.set_title("Boxplot pagal metus")
ax.set_xlabel("Metai")
ax.set_ylabel("Likana")
plt.suptitle("")

plt.suptitle("XGBoost likuciu diagnostika", fontsize=14)
plt.tight_layout()
plt.show()

#shapiro-wilk
if len(residuals) > 5000:
    rng = np.random.RandomState(SEED)
    sample = rng.choice(residuals, 5000, replace=False)
else:
    sample = residuals
sw_stat, sw_p = stats.shapiro(sample)
print(f"\nShapiro-Wilk: W={sw_stat:.4f}, p={sw_p:.6f}")
if sw_p < 0.05:
    print("  -> likuciai NEIRA normalus (p<0.05)")
else:
    print("  -> likuciai normalus (p>=0.05)")
